# Reaction-Diffusion Numerical Analysis

Numerical experiments for Gray-Scott and FitzHugh-Nagumo reaction-diffusion models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve2d
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown


## Gray-Scott model

In [ ]:
def make_stencil(w_direct, w_diag):
    return np.array([
        [w_diag,   w_direct, w_diag],
        [w_direct, -1.0,      w_direct],
        [w_diag,   w_direct,  w_diag]
    ])


def laplacian(Z, kernel):
    return convolve2d(Z, kernel, mode="same", boundary="wrap")


def init_gray_scott(N, mode="center", seed_size=5, density=0.05):
    A = np.ones((N, N))
    B = np.zeros((N, N))

    if mode == "center":
        c = N // 2
        A[c-seed_size:c+seed_size, c-seed_size:c+seed_size] = 0
        B[c-seed_size:c+seed_size, c-seed_size:c+seed_size] = 1
    else:
        mask = np.random.rand(N, N) < density
        A[mask] = 0
        B[mask] = 1

    return A, B


def run_gray_scott(A, B, kernel, Da, Db, f, k, dt, steps):
    for _ in range(steps):
        lapA = laplacian(A, kernel)
        lapB = laplacian(B, kernel)
        reaction = A * B * B

        A = np.clip(
            A + dt * (Da * lapA - reaction + f * (1 - A)),
            0, 1
        )
        B = np.clip(
            B + dt * (Db * lapB + reaction - (f + k) * B),
            0, 1
        )

    return A, B


In [ ]:
def simulate_gray_scott(
    w_direct, w_diag, f, k, Da, Db, dt, steps,
    mode, seed_size, density
):
    kernel = make_stencil(w_direct, w_diag)
    A, B = init_gray_scott(200, mode, seed_size, density)
    A, B = run_gray_scott(A, B, kernel, Da, Db, f, k, dt, steps)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(A, cmap="viridis")
    axes[0].set_title("A")
    axes[1].imshow(B, cmap="inferno")
    axes[1].set_title("B")
    axes[2].imshow(kernel, cmap="RdBu")
    axes[2].set_title("Stencil")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


interact(
    simulate_gray_scott,
    w_direct=FloatSlider(value=0.20, min=0.0, max=0.5, step=0.01),
    w_diag=FloatSlider(value=0.05, min=0.0, max=0.25, step=0.01),
    f=FloatSlider(value=0.035, min=0.01, max=0.1, step=0.001),
    k=FloatSlider(value=0.058, min=0.01, max=0.1, step=0.001),
    Da=FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1),
    Db=FloatSlider(value=0.5, min=0.1, max=2.0, step=0.1),
    dt=FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1),
    steps=IntSlider(value=1000, min=100, max=5000, step=100),
    mode=Dropdown(options=["center", "random"]),
    seed_size=IntSlider(value=5, min=1, max=30),
    density=FloatSlider(value=0.05, min=0.01, max=0.3, step=0.01),
)


## Gray-Scott parameter exploration

In [ ]:
def gray_scott_parameter_map(
    statistic="variance",
    N=50,
    steps=500,
    Da=1.0,
    Db=0.5,
    dt=1.0,
    grid_size=20
):
    f_values = np.linspace(0.01, 0.09, grid_size)
    k_values = np.linspace(0.01, 0.09, grid_size)
    results = np.zeros((len(f_values), len(k_values)))
    kernel = make_stencil(0.20, 0.05)

    for i, f in enumerate(f_values):
        for j, k in enumerate(k_values):
            A, B = init_gray_scott(N, "center", seed_size=5)
            A, B = run_gray_scott(
                A, B, kernel, Da, Db, f, k, dt, steps
            )

            results[i, j] = (
                np.var(B) if statistic == "variance" else np.mean(B)
            )

    label = "variance of B" if statistic == "variance" else "mean of B"

    plt.figure(figsize=(7, 6))
    plt.imshow(
        results,
        origin="lower",
        cmap="inferno",
        extent=[
            k_values[0], k_values[-1],
            f_values[0], f_values[-1]
        ]
    )
    plt.colorbar(label=label)
    plt.xlabel("k")
    plt.ylabel("f")
    plt.title(f"Gray-Scott parameter map ({label})")
    plt.show()

    return results


In [ ]:
variance_map = gray_scott_parameter_map(statistic="variance")
mean_map = gray_scott_parameter_map(statistic="mean")


## FitzHugh-Nagumo model

In [ ]:
def init_fitzhugh_nagumo(N, seed_size=5):
    U = np.full((N, N), -1.0)
    W = np.full((N, N), -1.0)

    c = N // 2
    U[c-seed_size:c+seed_size, c-seed_size:c+seed_size] = 1.0

    return U, W


def run_fitzhugh_nagumo(
    U, W, kernel, Da, Db, eps, p, q, dt, steps
):
    for _ in range(steps):
        lapU = laplacian(U, kernel)
        lapW = laplacian(W, kernel)

        U = np.clip(
            U + dt * (Da * lapU + U - U**3 - W),
            -2, 2
        )
        W = np.clip(
            W + dt * (Db * lapW + eps * (U + p - q * W)),
            -1, 2
        )

    return U, W


In [ ]:
def fitzhugh_nagumo_parameter_map(
    N=50,
    steps=500,
    Da=1.0,
    Db=20,
    eps=0.08,
    dt=0.05,
    grid_size=15
):
    kernel = make_stencil(0.20, 0.05)
    p_values = np.linspace(0, 1.5, grid_size)
    q_values = np.linspace(0, 1.5, grid_size)
    results = np.zeros((len(p_values), len(q_values)))

    total = len(p_values) * len(q_values)
    count = 0

    for i, p in enumerate(p_values):
        for j, q in enumerate(q_values):
            U, W = init_fitzhugh_nagumo(N, seed_size=3)
            U, W = run_fitzhugh_nagumo(
                U, W, kernel, Da, Db, eps, p, q, dt, steps
            )
            results[i, j] = np.var(U)

            count += 1
            print(f"{count}/{total}", end="\r")

    plt.figure(figsize=(6, 5))
    plt.imshow(
        results,
        origin="lower",
        cmap="inferno",
        extent=[
            q_values[0], q_values[-1],
            p_values[0], p_values[-1]
        ]
    )
    plt.colorbar(label="spatial variance of U")
    plt.xlabel("q")
    plt.ylabel("p")
    plt.title(f"FitzHugh-Nagumo parameter map (eps={eps})")
    plt.show()

    return results


In [ ]:
fhn_map = fitzhugh_nagumo_parameter_map(
    N=50,
    steps=1000,
    Da=1.0,
    Db=20,
    eps=0.08,
    dt=0.05
)


In [ ]:
p, q = 0.4, 0.4
kernel = make_stencil(0.20, 0.05)

U, W = init_fitzhugh_nagumo(100, seed_size=5)
U, W = run_fitzhugh_nagumo(
    U, W,
    kernel,
    Da=1.0,
    Db=20,
    eps=0.08,
    p=p,
    q=q,
    dt=0.05,
    steps=2000
)

plt.figure(figsize=(6, 6))
plt.imshow((U + 2) / 4, cmap="RdBu", vmin=0, vmax=1)
plt.title(f"FitzHugh-Nagumo: p={p}, q={q}")
plt.colorbar()
plt.show()
